# Experimento 5 — YOLOv8s a 1.024 pixels (TCC II)

Os experimentos 2 a 4 mostraram que, para a cárie, tanto o modelo maior (YOLOv8s) quanto a resolução maior (1.024) ajudam. Este notebook testa a combinação das duas mudanças, com o mesmo protocolo (seed 42, deterministic, batch 16, 100 épocas fixas, avaliação Ultralytics + pycocotools, best.pt e last.pt).

**Como rodar:** Settings → Accelerator **GPU T4 x2**, Internet **On** → **Save Version → Save & Run All (Commit)**. Estimativa: 1,5 a 2,5 h. No fim, baixe `artefatos_exp5.zip` na aba **Output** da versão salva.

Se faltar memória na GPU (erro *CUDA out of memory*), troque `batch=16` por `batch=8` na Célula 4 e registre a mudança.


In [ ]:
!pip install -q ultralytics==8.4.183 remotezip huggingface_hub pyyaml pycocotools


In [ ]:
# !pip install -q remotezip huggingface_hub pyyaml   (rode se faltar algum)
import json, shutil, time
from collections import Counter
from pathlib import Path, PurePosixPath

import yaml
from huggingface_hub import hf_hub_download, hf_hub_url
from remotezip import RemoteZip

REPO = "ibrahimhamamci/DENTEX"
OUT = Path("/kaggle/working/dentex_yolo")

# ---- Decisão metodológica em UM lugar só --------------------------------
# chave = category_id_3 original do DENTEX; valor = id da classe no YOLO (None = descartar)
CLASS_MAP = {0: 0, 1: 1, 2: 2, 3: 3}
YOLO_NAMES = {0: "impacted", 1: "caries", 2: "periapical_lesion", 3: "deep_caries"}
# Alternativas (basta trocar as duas linhas acima, sem rebaixar nada):
#  (b) sem impactado:       CLASS_MAP = {0: None, 1: 0, 2: 1, 3: 2}
#                           YOLO_NAMES = {0: "caries", 1: "periapical_lesion", 2: "deep_caries"}
#  (c) cáries unificadas:   CLASS_MAP = {0: 0, 1: 1, 2: 2, 3: 1}
#                           YOLO_NAMES = {0: "impacted", 1: "caries", 2: "periapical_lesion"}

# Pastas dentro dos ZIPs (atenção: o ZIP de treino tem 3 subconjuntos com nomes de arquivo parecidos)
TRAIN_ZIP, TRAIN_IMG_DIR = "training_data.zip", "training_data/quadrant-enumeration-disease/xrays/"
TRAIN_JSON = "training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"
VAL_ZIP, VAL_IMG_DIR = "validation_data.zip", "validation_data/quadrant_enumeration_disease/xrays/"


def coco_para_yolo(bbox, w_img, h_img):
    """COCO [x_min, y_min, largura, altura] em pixels -> YOLO [cx, cy, w, h] normalizado em [0, 1]."""
    x, y, w, h = bbox
    x0, y0 = max(0.0, x), max(0.0, y)                      # recorta caixas que escapam da imagem
    x1, y1 = min(float(w_img), x + w), min(float(h_img), y + h)
    if x1 <= x0 or y1 <= y0:
        return None
    return ((x0 + x1) / 2 / w_img, (y0 + y1) / 2 / h_img, (x1 - x0) / w_img, (y1 - y0) / h_img)


def converter(coco, zip_name, img_dir, split):
    """Extrai as imagens do split e grava um .txt de rótulos por imagem."""
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

    por_imagem = {}
    for a in coco["annotations"]:
        por_imagem.setdefault(a["image_id"], []).append(a)

    url = hf_hub_url(REPO, f"DENTEX/{zip_name}", repo_type="dataset")
    stats = Counter()
    with RemoteZip(url) as z:
        membros = {PurePosixPath(n).name: n for n in z.namelist() if n.startswith(img_dir) and n.endswith(".png")}
        for img in coco["images"]:
            nome = img["file_name"]
            destino = OUT / "images" / split / nome
            if not destino.exists():                        # idempotente: reexecutar não rebaixa
                for tentativa in range(3):
                    try:
                        with z.open(membros[nome]) as src, open(destino, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        break
                    except Exception as e:                  # rede instável: tenta de novo
                        print("retry", nome, e); time.sleep(2)
            linhas = []
            for a in por_imagem.get(img["id"], []):
                cls = CLASS_MAP[a["category_id_3"]]
                if cls is None:
                    stats["descartadas"] += 1
                    continue
                yolo = coco_para_yolo(a["bbox"], img["width"], img["height"])
                if yolo is None:
                    stats["caixas_invalidas"] += 1
                    continue
                linhas.append(f"{cls} " + " ".join(f"{v:.6f}" for v in yolo))
                stats[f"classe_{cls}"] += 1
            (OUT / "labels" / split / (Path(nome).stem + ".txt")).write_text("\n".join(linhas))  # vazio = fundo
            stats["imagens"] += 1
            stats["imagens_sem_caixa"] += (len(linhas) == 0)
    return stats


# ---- Treino ----
with RemoteZip(hf_hub_url(REPO, f"DENTEX/{TRAIN_ZIP}", repo_type="dataset")) as z:
    train_coco = json.loads(z.read(TRAIN_JSON))
val_coco = json.load(open(hf_hub_download(REPO, "DENTEX/validation_triple.json", repo_type="dataset")))

st_train = converter(train_coco, TRAIN_ZIP, TRAIN_IMG_DIR, "train")
st_val = converter(val_coco, VAL_ZIP, VAL_IMG_DIR, "val")
print("treino:", dict(st_train)); print("validação:", dict(st_val))

# ---- data.yaml: é ele que o YOLO lê no treino ----
(OUT / "data.yaml").write_text(yaml.safe_dump(
    {"path": str(OUT), "train": "images/train", "val": "images/val", "names": YOLO_NAMES},
    sort_keys=False, allow_unicode=True))
print((OUT / "data.yaml").read_text())
# Guarda o gabarito da validação em COCO: a avaliação com pycocotools (Célula 4) usa este arquivo.
(OUT / "val_coco.json").write_text(json.dumps(val_coco))


In [ ]:
from collections import Counter

def contar(split):
    c, vazios = Counter(), 0
    for f in (OUT / "labels" / split).glob("*.txt"):
        linhas = [l for l in f.read_text().splitlines() if l.strip()]
        vazios += (len(linhas) == 0)
        c.update(int(l.split()[0]) for l in linhas)
    return len(list((OUT / "images" / split).glob("*.png"))), vazios, dict(sorted(c.items()))

# Mesma conversão do baseline: se os números mudarem, PARE e investigue antes de comparar resultados.
esperado = {"train": (705, 27, {0: 604, 1: 2189, 2: 158, 3: 578}), "val": (50, 4, {0: 40, 1: 101, 2: 9, 3: 32})}
for s in ["train", "val"]:
    obtido = contar(s)
    print(s, "(imagens, sem caixa, caixas por classe):", obtido, "OK" if obtido == esperado[s] else "<<< DIFERENTE DO BASELINE")


In [ ]:
import json, shutil, time, traceback
from pathlib import Path

import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

SEED = 42
DATA = str(OUT / "data.yaml")
RUNS = Path("/kaggle/working/runs")
ART = Path("/kaggle/working/artefatos")          # só o que vai para o zip final
DEVICE = 0                                        # uma GPU, como no baseline
EPOCAS = 100

EXPERIMENTOS = {
    "exp2_yolov8n_640":  dict(modelo="yolov8n.pt", imgsz=640),
    "exp3_yolov8s_640":  dict(modelo="yolov8s.pt", imgsz=640),
    "exp4_yolov8n_1024": dict(modelo="yolov8n.pt", imgsz=1024),
    "exp5_yolov8s_1024": dict(modelo="yolov8s.pt", imgsz=1024),
}
RODAR = ["exp5_yolov8s_1024"]


def treinar(nome, modelo, imgsz):
    """Fine-tuning a partir dos pesos do COCO, com 100 épocas fixas (sem parada antecipada)."""
    YOLO(modelo).train(
        data=DATA, imgsz=imgsz, epochs=EPOCAS,
        patience=EPOCAS,          # paciência = nº de épocas -> a parada antecipada nunca dispara
        batch=16, seed=SEED, deterministic=True, device=DEVICE, workers=2,
        project=str(RUNS), name=nome, exist_ok=True,
    )


def avaliar_ultralytics(pesos, imgsz, nome_val):
    """Métricas do Ultralytics (mesmas do baseline). conf=0,001 é o padrão para mAP."""
    m = YOLO(pesos).val(data=DATA, imgsz=imgsz, split="val", device=DEVICE, plots=False,
                        project=str(RUNS), name=nome_val, exist_ok=True, verbose=False)
    linha = {"P": m.box.mp, "R": m.box.mr, "mAP50": m.box.map50, "mAP50-95": m.box.map}
    for k, c in enumerate(m.box.ap_class_index):
        linha[f"AP50_{m.names[c]}"] = m.box.ap50[k]
    return linha


def avaliar_coco(pesos, imgsz):
    """AP50 por classe com pycocotools, o protocolo do desafio DENTEX.

    1. Monta o gabarito COCO da validação com a classe de diagnóstico (category_id_3 -> CLASS_MAP).
    2. Prediz cada imagem (conf=0,001, como numa avaliação de mAP) e converte as caixas para
       [x, y, largura, altura] em pixels da imagem ORIGINAL.
    3. COCOeval compara tudo e devolve AP por classe e por limiar de IoU.
    """
    val = json.loads((OUT / "val_coco.json").read_text())
    gt = {"images": [{k: im[k] for k in ("id", "file_name", "width", "height")} for im in val["images"]],
          "categories": [{"id": i, "name": n} for i, n in YOLO_NAMES.items()],
          "annotations": []}
    for a in val["annotations"]:
        cls = CLASS_MAP[a["category_id_3"]]
        if cls is not None:
            x, y, w, h = a["bbox"]
            gt["annotations"].append({"id": len(gt["annotations"]) + 1, "image_id": a["image_id"],
                                      "category_id": cls, "bbox": [x, y, w, h], "area": w * h, "iscrowd": 0})
    id_por_nome = {im["file_name"]: im["id"] for im in gt["images"]}

    dets = []
    modelo = YOLO(pesos)
    for r in modelo.predict(source=str(OUT / "images" / "val"), imgsz=imgsz, conf=0.001, iou=0.7,
                            max_det=300, device=DEVICE, stream=True, verbose=False):
        img_id = id_por_nome[Path(r.path).name]
        for (x0, y0, x1, y1), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            dets.append({"image_id": img_id, "category_id": int(c),
                         "bbox": [x0, y0, x1 - x0, y1 - y0], "score": float(s)})

    if not dets:                                  # modelo sem nenhuma predição: não há o que avaliar
        return {"COCO_AP50": 0.0, "COCO_AP50-95": 0.0}
    coco_gt = COCO(); coco_gt.dataset = gt; coco_gt.createIndex()
    E = COCOeval(coco_gt, coco_gt.loadRes(dets), "bbox")
    E.evaluate(); E.accumulate(); E.summarize()
    # precision[T, R, K, A, M]: T=limiar de IoU (0 -> 0,50), K=classe, A=0 (todas as áreas), M=2 (100 det.)
    prec = E.eval["precision"]
    linha = {"COCO_AP50": E.stats[1], "COCO_AP50-95": E.stats[0]}
    for k, cid in enumerate(E.params.catIds):
        p = prec[0, :, k, 0, 2]
        linha[f"COCO_AP50_{YOLO_NAMES[cid]}"] = float(p[p > -1].mean()) if (p > -1).any() else float("nan")
    return linha


def matriz_conf025(pesos, imgsz, destino, prefixo):
    """Matriz de confusão num limiar realista (0,25), só para figura/análise de erros."""
    nome = f"val025_{prefixo}"
    YOLO(pesos).val(data=DATA, imgsz=imgsz, split="val", device=DEVICE, conf=0.25, plots=True,
                    project=str(RUNS), name=nome, exist_ok=True, verbose=False)
    for n in ["confusion_matrix.png", "confusion_matrix_normalized.png"]:
        if (RUNS / nome / n).exists():
            shutil.copy(RUNS / nome / n, destino / f"{prefixo}_{n.replace('.png', '_conf025.png')}")


def resumo_curvas(nome):
    """Números do treino tirados do results.csv (por época)."""
    df = pd.read_csv(RUNS / nome / "results.csv"); df.columns = [c.strip() for c in df.columns]
    col = "metrics/mAP50-95(B)"
    return {"epocas": len(df), "melhor_epoca": int(df[col].idxmax()) + 1,
            "mAP50-95_pico": df[col].max(),
            "mAP50-95_media_ult10": df[col].tail(10).mean(),   # platô: número menos sortudo que o pico
            "horas": df["time"].iloc[-1] / 3600 if "time" in df else float("nan")}


def rodar(nome):
    cfg = EXPERIMENTOS[nome]
    destino = ART / nome; destino.mkdir(parents=True, exist_ok=True)
    pesos_dir = RUNS / nome / "weights"
    if not (pesos_dir / "last.pt").exists():
        treinar(nome, **cfg)
    linhas = []
    for qual in ["best", "last"]:
        pesos = str(pesos_dir / f"{qual}.pt")
        linha = {"experimento": nome, "pesos": qual, **cfg, **resumo_curvas(nome)}
        linha.update(avaliar_ultralytics(pesos, cfg["imgsz"], f"val_{nome}_{qual}"))
        linha.update(avaliar_coco(pesos, cfg["imgsz"]))
        matriz_conf025(pesos, cfg["imgsz"], destino, f"{nome}_{qual}")
        linhas.append(linha)
    for n in ["results.csv", "results.png", "args.yaml", "BoxPR_curve.png", "weights/best.pt", "weights/last.pt"]:
        if (RUNS / nome / n).exists():
            shutil.copy(RUNS / nome / n, destino / Path(n).name)
    pd.DataFrame(linhas).to_csv(destino / "metricas.csv", index=False)
    return linhas


In [ ]:
# Roda cada experimento; se um falhar, registra o erro e segue para o próximo.
ART.mkdir(parents=True, exist_ok=True)
todas, erros = [], {}
for nome in RODAR:
    t0 = time.time()
    print(f"\n{'=' * 70}\n{nome}\n{'=' * 70}")
    try:
        todas += rodar(nome)
        pd.DataFrame(todas).to_csv(ART / "resumo.csv", index=False)   # salva a cada experimento
    except Exception:
        erros[nome] = traceback.format_exc(); print(erros[nome])
    print(f"{nome}: {(time.time() - t0) / 3600:.2f} h")
(ART / "erros.json").write_text(json.dumps(erros, indent=2))
print("Erros:", list(erros) or "nenhum")


In [ ]:
import matplotlib.pyplot as plt

# Referência: baseline YOLOv8n (patience 20, parou na época 46), números do Ultralytics no best.pt.
BASELINE = {"experimento": "exp1_baseline_yolov8n_640", "pesos": "best", "P": 0.473, "R": 0.640,
            "mAP50": 0.582, "AP50_impacted": 0.916, "AP50_caries": 0.318,
            "AP50_periapical_lesion": 0.508, "AP50_deep_caries": 0.587}

EXP3 = {"experimento": "exp3_yolov8s_640", "pesos": "best", "P": 0.593, "R": 0.566, "mAP50": 0.553,
        "mAP50-95_media_ult10": 0.337, "AP50_caries": 0.386, "COCO_AP50": 0.547, "COCO_AP50_caries": 0.392}
res = pd.concat([pd.DataFrame([BASELINE, EXP3]), pd.read_csv(ART / "resumo.csv")], ignore_index=True)
cols = ["experimento", "pesos", "epocas", "melhor_epoca", "mAP50-95_pico", "mAP50-95_media_ult10",
        "P", "R", "mAP50", "AP50_caries", "AP50_deep_caries", "AP50_periapical_lesion", "AP50_impacted",
        "COCO_AP50", "COCO_AP50_caries"]
pd.set_option("display.width", 250)
print(res[[c for c in cols if c in res]].round(3).to_string(index=False))

# Figura: mAP50-95 por época de cada experimento (mostra o ruído da validação e o platô)
fig, ax = plt.subplots(figsize=(10, 5))
for nome in RODAR:
    f = RUNS / nome / "results.csv"
    if f.exists():
        df = pd.read_csv(f); df.columns = [c.strip() for c in df.columns]
        ax.plot(df["epoch"], df["metrics/mAP50-95(B)"], label=nome, lw=1.2)
ax.set_xlabel("Época"); ax.set_ylabel("mAP50-95 (validação, 50 imagens)"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(ART / "curvas_map50-95.png", dpi=150); plt.show()


In [ ]:
# Última célula: apaga a base convertida (é recriada a cada execução) para a saída ficar leve, e gera o zip.
shutil.rmtree(OUT, ignore_errors=True)
shutil.make_archive("/kaggle/working/artefatos_exp5", "zip", ART)
print(sorted(str(p.relative_to(ART)) for p in ART.rglob("*") if p.is_file()))
